In [ ]:
# -*- coding: utf-8 -*-
"""
Comparação visual direta: Park vs RF ponto-a-ponto (lado a lado)
- Carrega a mesma base (.pkl) uma única vez
- Calcula referência (falha=0) em REF_TEMP (ou fallback por mediana)
- Compensa UMA curva escolhida com Park (shift por amostras + offset + suavização)
- Treina RF regressão ponto-a-ponto usando apenas amostras sem falha
- Aplica RF na mesma curva escolhida
- Plota Park vs RF em 1x2, estilo artigo, mais clean

Autor: Luiz Eduardo Abdala José (consolidado)
"""

import os, re, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor

warnings.filterwarnings("ignore", category=UserWarning)

# ================================================================
# ========================= PARÂMETROS ============================
# ================================================================
ARQ_BASE = "base-completo--.pkl"

REF_TEMP = 30
TEMP_DESEJADA = 48
FALHA_DESEJADA = None

FREQ_MIN_KHZ = 40
FREQ_MAX_KHZ = 50

# -------------------- PARK (1999) --------------------
PARK_OVERLAP_MIN = 0.60
PARK_SMOOTH_WIN  = 5
PARK_NSHIFTS     = 401

# -------------------- RF COMP (ponto-a-ponto) --------
RF_COMP_PARAMS = dict(
    n_estimators=400,
    max_depth=10,
    min_samples_leaf=2,
    min_samples_split=4,
    max_features="sqrt",
    n_jobs=-1,
    random_state=0
)
SMOOTH_WIN_RF = 6


# ================================================================
# ===================== FUNÇÕES AUXILIARES =======================
# ================================================================
def extract_freq_hz(col):
    m = re.match(r"^f_(\d+(?:\.\d+)?)Hz$", str(col))
    return float(m.group(1)) if m else None

def get_freq_columns(df, fmin_khz, fmax_khz):
    cols, freqs = [], []
    for c in df.columns:
        f = extract_freq_hz(c)
        if f is not None and fmin_khz <= f/1e3 <= fmax_khz:
            cols.append(c)
            freqs.append(f)
    order = np.argsort(freqs)
    return [cols[i] for i in order], np.array(freqs, float)[order]

def moving_average(arr, win):
    if win <= 1 or win % 2 == 0:
        return np.asarray(arr).copy()
    pad = win // 2
    arr = np.asarray(arr, float)
    arr_pad = np.pad(arr, (pad, pad), mode="edge")
    kernel = np.ones(win, float) / win
    out = np.convolve(arr_pad, kernel, mode="valid")
    return out[:len(arr)]

def is_uniform_grid(fhz, rtol=1e-4, atol=1e-9):
    d = np.diff(fhz)
    return np.allclose(d, d[0], rtol=rtol, atol=atol)

def shift_by_samples(x, k):
    x = np.asarray(x, float)
    n = len(x)
    if k == 0:
        return x.copy()
    y = np.empty_like(x)
    if k > 0:
        y[:k] = x[0]
        y[k:] = x[:n-k]
    else:
        kk = -k
        y[n-kk:] = x[-1]
        y[:n-kk] = x[kk:]
    return y

def park_compensate_single_sampleshift(x, y_ref,
                                      overlap_min_frac=PARK_OVERLAP_MIN,
                                      smooth_win=PARK_SMOOTH_WIN,
                                      nshifts=PARK_NSHIFTS):
    x = np.asarray(x, float)
    y_ref = np.asarray(y_ref, float)
    n = len(x)

    if n < 5:
        return x.copy(), 0, 0.0

    k_max = int(np.floor((1.0 - overlap_min_frac) * (n - 1)))
    k_max = max(k_max, 0)

    if k_max == 0:
        ks = np.array([0], dtype=int)
    else:
        ks = np.linspace(-k_max, +k_max, int(nshifts)).round().astype(int)
        ks = np.unique(ks)

    best_Va = np.inf
    best_k = 0
    best_dS = 0.0

    for k in ks:
        xs = shift_by_samples(x, int(k))
        dS = float(np.mean(y_ref - xs))
        r = y_ref - (xs + dS)
        Va = float(np.sum(r * r))
        if Va < best_Va:
            best_Va = Va
            best_k = int(k)
            best_dS = float(dS)

    ycorr = shift_by_samples(x, best_k) + best_dS

    if smooth_win > 1:
        ycorr = moving_average(ycorr, smooth_win)

    return ycorr, best_k, best_dS

def add_extra_features(X):
    X = np.asarray(X, float)
    mu = X.mean(axis=1, keepdims=True)
    sd = X.std(axis=1, keepdims=True)
    amp = (X.max(axis=1) - X.min(axis=1)).reshape(-1, 1)
    return np.hstack([X, mu, sd, amp])

def add_temp_feature(X_aug, temp_vec):
    return np.hstack([X_aug, np.asarray(temp_vec, float).reshape(-1, 1)])


# ================================================================
# ============================ SCRIPT =============================
# ================================================================
if not os.path.exists(ARQ_BASE):
    raise FileNotFoundError(
        f"Não encontrei '{ARQ_BASE}'. Coloque o .pkl na mesma pasta ou use caminho absoluto."
    )

# 1) Carrega base e seleciona faixa de frequência
df = pd.read_pickle(ARQ_BASE)
fcols, fhz = get_freq_columns(df, FREQ_MIN_KHZ, FREQ_MAX_KHZ)
fhz_khz = fhz / 1e3

if not is_uniform_grid(fhz):
    raise ValueError(
        "A malha de frequência não parece uniforme. Park por shift inteiro pode distorcer."
    )

# 2) Define referência (falha=0)
df_sem = df[df["falha"] == 0].copy()
pool_ref = df_sem.loc[np.isclose(df_sem["temperatura_c"], REF_TEMP), fcols].to_numpy(float)

if len(pool_ref) > 0:
    y_ref = np.median(pool_ref, axis=0)
else:
    y_ref = np.median(df_sem[fcols].to_numpy(float), axis=0)

# 3) Escolhe a curva para comparação
df_sel = df[np.isclose(df["temperatura_c"], TEMP_DESEJADA)].copy()
if FALHA_DESEJADA is not None:
    df_sel = df_sel[df_sel["falha"] == int(FALHA_DESEJADA)]

if len(df_sel) == 0:
    raise ValueError(
        f"Não encontrei amostras em TEMP_DESEJADA={TEMP_DESEJADA}°C"
        + (f" com falha={FALHA_DESEJADA}." if FALHA_DESEJADA is not None else ".")
    )

idx_show = df_sel.index[0]
y_orig = df.loc[idx_show, fcols].to_numpy(float)
temp_real = float(df.loc[idx_show, "temperatura_c"])
falha_real = int(df.loc[idx_show, "falha"])

# 4) PARK
y_park, k_best, dS_best = park_compensate_single_sampleshift(
    y_orig, y_ref,
    overlap_min_frac=PARK_OVERLAP_MIN,
    smooth_win=PARK_SMOOTH_WIN,
    nshifts=PARK_NSHIFTS
)

# 5) RF ponto-a-ponto
X_sem = df_sem[fcols].to_numpy(float)
T_sem = df_sem["temperatura_c"].to_numpy(float)

Y_target = y_ref[None, :] - X_sem
X_aug = add_extra_features(X_sem)
X_comp = add_temp_feature(X_aug, T_sem)

rf_comp = RandomForestRegressor(**RF_COMP_PARAMS).fit(X_comp, Y_target)

# 6) Aplica RF
X_one = y_orig[None, :]
T_one = np.array([temp_real], float)

X_one_aug = add_extra_features(X_one)
X_one_comp = add_temp_feature(X_one_aug, T_one)

delta_hat = rf_comp.predict(X_one_comp)[0]
y_rf = y_orig + delta_hat

if SMOOTH_WIN_RF > 1:
    y_rf = moving_average(y_rf, SMOOTH_WIN_RF)


# ================================================================
# ============================ PLOT 1x2 ===========================
# ================================================================
plt.rcParams.update({
    "font.family": "serif",
    "font.size": 13,
    "axes.titlesize": 14,
    "axes.labelsize": 13,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "legend.fontsize": 12,
    "text.usetex": False
})

fig, axes = plt.subplots(
    1, 2,
    figsize=(15.5, 5.8),
    sharex=True,
    sharey=True,
    dpi=300
)

fig.patch.set_facecolor("white")

for ax in axes:
    ax.grid(False)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_linewidth(0.8)
    ax.spines["bottom"].set_linewidth(0.8)
    ax.tick_params(axis="both", which="major", length=4, width=0.8)

# -------------------- (a) PARK --------------------
ax = axes[0]
l1, = ax.plot(
    fhz_khz, y_ref,
    "--", c="black", lw=1.1,
    label=f"Referência {REF_TEMP}°C"
)
l2, = ax.plot(
    fhz_khz, y_orig,
    c="tab:red", lw=1.4, alpha=0.55,
    label=f"Original {temp_real:.0f}°C"
)
l3, = ax.plot(
    fhz_khz, y_park,
    c="tab:blue", lw=2.0,
    label="Park"
)

ax.set_title("(a) Compensação por Park", pad=10)
ax.set_xlabel("Frequência (kHz)")
ax.set_ylabel("Parte real da impedância")

# -------------------- (b) RF --------------------
ax = axes[1]
ax.plot(
    fhz_khz, y_ref,
    "--", c="black", lw=1.1,
    label=f"Referência {REF_TEMP}°C"
)
ax.plot(
    fhz_khz, y_orig,
    c="tab:red", lw=1.4, alpha=0.55,
    label=f"Original {temp_real:.0f}°C"
)
ax.plot(
    fhz_khz, y_rf,
    c="tab:green", lw=2.0,
    label="RF ponto-a-ponto"
)

ax.set_title("(b) Compensação por RF ponto-a-ponto", pad=10)
ax.set_xlabel("Frequência (kHz)")

# -------------------- legenda global --------------------
handles = [
    l1,
    l2,
    plt.Line2D([0], [0], color="tab:blue", lw=2.0, label="Park"),
    plt.Line2D([0], [0], color="tab:green", lw=2.0, label="RF ponto-a-ponto")
]
labels = [h.get_label() for h in handles]

fig.legend(
    handles, labels,
    loc="upper center",
    ncol=4,
    frameon=False,
    bbox_to_anchor=(0.5, 1.03),
    handlelength=2.2,
    columnspacing=1.6,
    fontsize=12.5
)


plt.tight_layout(rect=[0, 0, 1, 0.92])
plt.show()

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams.update({
    "font.family": "Times New Roman",
    "font.size": 22,
    "axes.labelsize": 22,
    "xtick.labelsize": 20,
    "ytick.labelsize": 20,
    "legend.fontsize": 20,
})

fig, axes = plt.subplots(
    1, 2,
    figsize=(17, 7.2),
    sharex=True,
    sharey=True,
    dpi=300
)

fig.patch.set_facecolor("white")

for ax in axes:
    ax.grid(False)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_linewidth(1.0)
    ax.spines["bottom"].set_linewidth(1.0)
    ax.tick_params(axis="both", length=5, width=1)

# ======================= (a) Park =======================

ax = axes[0]

l1, = ax.plot(
    fhz_khz, y_ref,
    "--", color="black", lw=1.6,
    label=f"Reference {REF_TEMP}°C"
)

l2, = ax.plot(
    fhz_khz, y_orig,
    color="tab:red", lw=2.0, alpha=0.55,
    label=f"Original {temp_real:.0f}°C"
)

l3, = ax.plot(
    fhz_khz, y_park,
    color="tab:blue", lw=2.6,
    label="Park"
)

ax.set_xlabel("Frequency (kHz)", labelpad=10)
ax.set_ylabel("Real component of impedance")

ax.text(
    0.5, -0.30,
    "(a) Park",
    transform=ax.transAxes,
    ha="center",
    fontsize=22
)

# ======================= (b) RF =======================

ax = axes[1]

ax.plot(
    fhz_khz, y_ref,
    "--", color="black", lw=1.6
)

ax.plot(
    fhz_khz, y_orig,
    color="tab:red", lw=2.0, alpha=0.55
)

ax.plot(
    fhz_khz, y_rf,
    color="tab:green", lw=2.6
)

ax.set_xlabel("Frequency (kHz)", labelpad=10)

ax.text(
    0.5, -0.30,
    "(b) Random Forest point-by-point",
    transform=ax.transAxes,
    ha="center",
    fontsize=22
)

handles = [
    l1,
    l2,
    plt.Line2D([0], [0], color="tab:blue", lw=2.6),
    plt.Line2D([0], [0], color="tab:green", lw=2.6)
]

labels = [
    f"Reference {REF_TEMP}°C",
    f"Original {temp_real:.0f}°C",
    "Park",
    "Random Forest point-by-point"
]

fig.legend(
    handles,
    labels,
    loc="upper center",
    ncol=4,
    frameon=False,
    bbox_to_anchor=(0.5, 1.06),
    columnspacing=2,
    handlelength=2.6
)

plt.tight_layout(rect=[0, 0.10, 1, 0.92])
plt.show()

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams.update({
    "font.family": "Times New Roman",
    "font.size": 22,
    "axes.labelsize": 22,
    "xtick.labelsize": 20,
    "ytick.labelsize": 20,
    "legend.fontsize": 20,
})

fig, axes = plt.subplots(
    1, 2,
    figsize=(17, 7.2),
    sharex=True,
    sharey=True,
    dpi=300
)

fig.patch.set_facecolor("white")

for ax in axes:
    ax.grid(False)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_linewidth(1.0)
    ax.spines["bottom"].set_linewidth(1.0)
    ax.tick_params(axis="both", length=5, width=1)

# ======================= (a) Park =======================

ax = axes[0]

l1, = ax.plot(
    fhz_khz, y_ref,
    "--", color="black", lw=1.6,
    label=f"Reference {REF_TEMP}°C"
)

l2, = ax.plot(
    fhz_khz, y_orig,
    color="tab:red", lw=2.0, alpha=0.55,
    label=f"Original {temp_real:.0f}°C"
)

l3, = ax.plot(
    fhz_khz, y_park,
    color="tab:blue", lw=2.6,
    label="Park"
)

ax.set_xlabel("Frequency (kHz)", labelpad=10)
ax.set_ylabel("Real component of impedance")

ax.text(
    0.5, -0.30,
    "(a) Park",
    transform=ax.transAxes,
    ha="center",
    fontsize=22
)

# ======================= (b) RF =======================

ax = axes[1]

ax.plot(
    fhz_khz, y_ref,
    "--", color="black", lw=1.6
)

ax.plot(
    fhz_khz, y_orig,
    color="tab:red", lw=2.0, alpha=0.55
)

ax.plot(
    fhz_khz, y_rf,
    color="tab:green", lw=2.6
)

ax.set_xlabel("Frequency (kHz)", labelpad=10)

ax.text(
    0.5, -0.30,
    "(b) Random Forest point-by-point",
    transform=ax.transAxes,
    ha="center",
    fontsize=22
)

handles = [
    l1,
    l2,
    plt.Line2D([0], [0], color="tab:blue", lw=2.6),
    plt.Line2D([0], [0], color="tab:green", lw=2.6)
]

labels = [
    f"Reference {REF_TEMP}°C",
    f"Original {temp_real:.0f}°C",
    "Park",
    "Random Forest point-by-point"
]

fig.legend(
    handles,
    labels,
    loc="upper center",
    ncol=4,
    frameon=False,
    bbox_to_anchor=(0.5, 1.06),
    columnspacing=2,
    handlelength=2.6
)

plt.tight_layout(rect=[0, 0.10, 1, 0.92])

# ======================= SALVAR PDF =======================
plt.savefig(
    "comparacao_metodos.pdf",
    format="pdf",
    bbox_inches="tight"
)

plt.show()

In [ ]:
plt.savefig


In [ ]:
import os
import matplotlib.pyplot as plt

def _ensure_output_dir(output_dir):
    os.makedirs(output_dir, exist_ok=True)

def _save_figure(fig, output_dir, base_name, dpi=600):
    _ensure_output_dir(output_dir)

    pdf_path = os.path.join(output_dir, f"{base_name}.pdf")
    png_path = os.path.join(output_dir, f"{base_name}.png")

    fig.savefig(pdf_path, bbox_inches="tight")
    fig.savefig(png_path, dpi=dpi, bbox_inches="tight")

    return pdf_path, png_path


plt.rcParams.update({
    "font.family": "Times New Roman",
    "font.size": 22,
    "axes.labelsize": 22,
    "xtick.labelsize": 20,
    "ytick.labelsize": 20,
    "legend.fontsize": 20,
})

fig, axes = plt.subplots(
    1, 2,
    figsize=(17, 7.2),
    sharex=True,
    sharey=True,
    dpi=300
)

fig.patch.set_facecolor("white")

for ax in axes:
    ax.grid(False)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_linewidth(1.0)
    ax.spines["bottom"].set_linewidth(1.0)
    ax.tick_params(axis="both", length=5, width=1)

# ======================= (a) Park =======================

ax = axes[0]

l1, = ax.plot(
    fhz_khz, y_ref,
    "--", color="black", lw=1.6,
    label=f"Reference {REF_TEMP}°C"
)

l2, = ax.plot(
    fhz_khz, y_orig,
    color="tab:red", lw=2.0, alpha=0.55,
    label=f"Original {temp_real:.0f}°C"
)

l3, = ax.plot(
    fhz_khz, y_park,
    color="tab:blue", lw=2.6,
    label="Park"
)

ax.set_xlabel("Frequency (kHz)", labelpad=10)
ax.set_ylabel("Real component of impedance")

ax.text(
    0.5, -0.30,
    "(a) Park",
    transform=ax.transAxes,
    ha="center",
    fontsize=22
)

# ======================= (b) RF =======================

ax = axes[1]

ax.plot(
    fhz_khz, y_ref,
    "--", color="black", lw=1.6
)

ax.plot(
    fhz_khz, y_orig,
    color="tab:red", lw=2.0, alpha=0.55
)

ax.plot(
    fhz_khz, y_rf,
    color="tab:green", lw=2.6
)

ax.set_xlabel("Frequency (kHz)", labelpad=10)

ax.text(
    0.5, -0.30,
    "(b) Random Forest point-by-point",
    transform=ax.transAxes,
    ha="center",
    fontsize=22
)

handles = [
    l1,
    l2,
    plt.Line2D([0], [0], color="tab:blue", lw=2.6),
    plt.Line2D([0], [0], color="tab:green", lw=2.6)
]

labels = [
    f"Reference {REF_TEMP}°C",
    f"Original {temp_real:.0f}°C",
    "Park",
    "Random Forest point-by-point"
]

fig.legend(
    handles,
    labels,
    loc="upper center",
    ncol=4,
    frameon=False,
    bbox_to_anchor=(0.5, 1.06),
    columnspacing=2,
    handlelength=2.6
)

plt.tight_layout(rect=[0, 0.10, 1, 0.92])

pdf_path, png_path = _save_figure(
    fig,
    output_dir="figuras",
    base_name=f"comparacao_park_rf_{temp_real:.0f}C",
    dpi=600
)

plt.show()

print(f"Figura salva em PDF: {pdf_path}")
print(f"Figura salva em PNG: {png_path}")